# LEAN Backtest Runner

Use this notebook to rerun a LEAN project from Python without touching the other test jobs that may still be running.

It assumes:
- `pip install lean` has already been run
- Docker is available and running
- you have already done `lean login` and `lean init` in the target workspace
- `LEAN_WORKSPACE_DIR` points at the LEAN workspace root
- `LEAN_PROJECT_NAME` is the project folder name inside that workspace


In [ ]:
from __future__ import annotations

import os
import shlex
import shutil
import subprocess
from pathlib import Path

LEAN_WORKSPACE_DIR = Path(os.environ.get("LEAN_WORKSPACE_DIR", "/home/jlee153232/PycharmProjects/lean_workspace"))
LEAN_PROJECT_NAME = os.environ.get("LEAN_PROJECT_NAME", "lean_option_backtest")
LEAN_REPORT_PATH = LEAN_WORKSPACE_DIR / LEAN_PROJECT_NAME / "report.html"

def run_cmd(cmd: list[str], *, cwd: Path) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(shlex.quote(part) for part in cmd))
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr)
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    return proc

print("LEAN binary:", shutil.which("lean"))
print("Workspace:", LEAN_WORKSPACE_DIR)
print("Project:", LEAN_PROJECT_NAME)
print("Report:", LEAN_REPORT_PATH)


In [ ]:
run_cmd(["lean", "--version"], cwd=LEAN_WORKSPACE_DIR)


In [ ]:
backtest_cmd = ["lean", "backtest", LEAN_PROJECT_NAME]
result = run_cmd(backtest_cmd, cwd=LEAN_WORKSPACE_DIR)
print("Backtest completed.")


In [ ]:
if LEAN_REPORT_PATH.exists():
    print(f"LEAN report created at {LEAN_REPORT_PATH}")
else:
    print("LEAN report not found yet. Run `lean report` in the workspace if you want the HTML report.")

try:
    run_cmd(["lean", "report"], cwd=LEAN_WORKSPACE_DIR)
except Exception as exc:
    print(f"Report generation skipped or failed: {exc}")
